# GeoCPU v48｜40,500からの追加学習（A100・全教材継承）

**既存の423,348行（従来422,500行＋v47追加848行）を1行も削らずに継承**し、新規4,096問（4分野×1,024）を追加します。親はstep 40,500のモデル・Adam・乱数状態です。元の保存物は上書きせず、新しいフォルダーへ保存します。

実行は2段階です。**①はCPUランタイム、②はA100ランタイム**で動かします。

| 段階 | ランタイム | 内容 |
|---|---|---|
| ① CPU事前検査 | CPU（GPU不要） | 423,348行をハッシュで照合し、v48の4,096問を加える。旧教材の**全ソース**について、40,500段階と同じ条件で教師トークン数が減らない最小の更新回数を、サンプラーの1回の試算で決める。結果をSHA256つきでDriveに保存する |
| ② A100学習 | A100 40GB以上 | ①の結果を照合して再利用する（同じ巨大な検査を繰り返さない）。①で表示された更新回数を`CONFIRM_UPDATES`に書いた場合だけ学習を始める |

- 学習設定は40,500段階と同じです。36,864トークン/更新、バッチ上限64、学習率3×10⁻⁶、BF16、凍結した親を教師にしたKL。
- 旧ソースの教師トークンが足りない場合は、そのソースの行を**明示した重みで反復**して補います。反復した行は独立した問題数に数えません。
- 一度作った30500・40500の学習データは、SHA256を照合したうえで`_v48_dataset_cache/`から再利用します。2回目以降の①は数分で終わります。

コードはGitHubの研究ブランチ`fix/v48-cpu-exposure-rebalance-20261108`（PR #7）から取得します。マージ後は`BRANCH = "main"`に変えてください。

In [ ]:
# ① CPU事前検査（ランタイムのタイプ: CPU。GPUは使いません）
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, tempfile

BRANCH = "fix/v48-cpu-exposure-rebalance-20261108"   # PR #7。マージ後は "main"
drive.mount("/content/drive")
REPO = Path(tempfile.mkdtemp(prefix="geocpu_v48_repo_", dir="/content")) / "geocpu-diagnostics"
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/corcondor/geocpu-diagnostics.git", str(REPO)], check=True)
RUNNER = REPO / "research/v48_balanced_augmentation_20261108/train_from_40500.py"
assert "--gpu-profile" in RUNNER.read_text(encoding="utf-8"), "このブランチには A100 版の事前検査がありません"

STUDY = Path("/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106")
ARCHIVE = STUDY / "GeoCPU_practical_training_logging_20261107" / "GeoCPU_practical_training_logging_20261107_10k.zip"
assert ARCHIVE.is_file(), f"元v47 ZIPが見つかりません: {ARCHIVE}"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = STUDY / "_v48_preflight" / f"preflight_a100_{STAMP}.json"
LOG = REPORT.with_suffix(".log")
REPORT.parent.mkdir(parents=True, exist_ok=True)
# 事前検査だけなので出力先フォルダーは作られません（存在しない新しい名前を渡すだけ）。
OUT_UNUSED = STUDY / f"v48_A100_preflight_only_{STAMP}"

cmd = [sys.executable, "-u", str(RUNNER), "--study", str(STUDY), "--v47-zip", str(ARCHIVE),
       "--output", str(OUT_UNUSED), "--gpu-profile", "a100", "--preflight-report", str(REPORT)]
tail = []
with LOG.open("x", encoding="utf-8") as log, subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1) as child:
    for line in child.stdout:
        log.write(line); log.flush()
        tail = (tail + [line.rstrip()])[-80:]
        if line.startswith(("V48_", "Traceback", "[fix_v43]", "[build_parent]", "[compose]", "[prepare]")) or "Error" in line:
            print(line, end="", flush=True)
    code = child.wait()
if code != 0:
    print("\n".join(tail))
    raise RuntimeError(f"CPU事前検査が失敗しました（終了コード{code}）。GPUは使っていません。ログ: {LOG}")

r = json.loads(REPORT.read_text(encoding="utf-8"))
rec = r["records"]
print("\n===== CPU事前検査 通過 =====")
print("承認ファイル:", REPORT)
print(f"継承した既存行: {rec['inherited_rows']:,}（SHA256 {r['parent_train_sha256'][:16]}… と一致）")
print(f"本当に新しい問題: {rec['new_unique_examples']:,}")
print(f"新分野の反復行: {rec['new_domain_repeat_rows']:,} ／ 旧ソースの補充反復行: {rec['old_source_replay_replica_rows']:,}")
print(f"学習ファイルの行数: {rec['training_file_rows']:,}（独立した問題は {rec['independent_problems']:,}）")
print("旧ソースの補充重み:", r["train"].get("replay_source_training_weights") or "なし")
print(f"必要な更新回数: {r['steps']:,}（step {r['start_step']:,} → {r['stop_step']:,}）")
print(f"A100の見込み時間: 約{r['estimated_gpu_hours']}時間（40,500段階の実測 1.27更新/秒から）")
low = {k: v for k, v in r["old_source_floors"].items() if v["ratio"] is not None and v["ratio"] < 1.0}
print("40,500段階より教師トークンが減る旧ソース:", low or "なし（全ソース維持）")
print(f"\n② に進む場合は CONFIRM_UPDATES = {r['steps']} と書いてください。")


In [ ]:
# ② A100学習（ランタイムのタイプ: A100）。①の結果を再利用し、同じ検査は繰り返しません。
APPROVED = None          # 空なら _v48_preflight/ の最新の承認ファイルを使います
CONFIRM_UPDATES = None   # ①で表示された「必要な更新回数」をそのまま書く（GPU時間の確認）

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, tempfile
import torch

BRANCH = "fix/v48-cpu-exposure-rebalance-20261108"   # PR #7。マージ後は "main"
drive.mount("/content/drive")
assert torch.cuda.is_available(), "ランタイムのタイプで A100 GPU を選んでください"
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
assert "A100" in GPU and VRAM >= 39, f"A100 40GB以上が必要です: {GPU} {VRAM:.1f}GiB"
STUDY = Path("/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106")
reports = sorted((STUDY / "_v48_preflight").glob("preflight_a100_*.json"))
APPROVED = Path(APPROVED) if APPROVED else (reports[-1] if reports else None)
assert APPROVED and APPROVED.is_file(), "先に ① CPU事前検査 を通してください"
plan = json.loads(APPROVED.read_text(encoding="utf-8"))
print("承認ファイル:", APPROVED)
print(f"更新回数 {plan['steps']:,}／見込み 約{plan['estimated_gpu_hours']}時間／GPU {GPU} {VRAM:.0f}GiB")
assert CONFIRM_UPDATES == plan["steps"], f"CONFIRM_UPDATES = {plan['steps']} と書いてから実行してください（GPU時間の確認）"

REPO = Path(tempfile.mkdtemp(prefix="geocpu_v48_repo_", dir="/content")) / "geocpu-diagnostics"
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/corcondor/geocpu-diagnostics.git", str(REPO)], check=True)
RUNNER = REPO / "research/v48_balanced_augmentation_20261108/train_from_40500.py"
ARCHIVE = STUDY / "GeoCPU_practical_training_logging_20261107" / "GeoCPU_practical_training_logging_20261107_10k.zip"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = STUDY / f"v48_A100_from40500_{STAMP}"
LOG = STUDY / "_v48_preflight" / f"train_a100_{STAMP}.log"
cmd = [sys.executable, "-u", str(RUNNER), "--study", str(STUDY), "--v47-zip", str(ARCHIVE),
       "--output", str(OUT), "--gpu-profile", "a100", "--approved-preflight", str(APPROVED),
       "--confirm-updates", str(CONFIRM_UPDATES), "--execute"]
tail, seen = [], 0
with LOG.open("x", encoding="utf-8") as log, subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1) as child:
    for line in child.stdout:
        log.write(line); seen += 1
        tail = (tail + [line.rstrip()])[-80:]
        if seen % 20 == 0:
            log.flush()
        if line.startswith(("V48_", "Traceback", "[A100]")) or "Step " in line or "Error" in line:
            print(line, end="", flush=True)
    code = child.wait()
if code != 0:
    print("\n".join(tail))
    raise RuntimeError(f"学習が終了コード{code}で止まりました。ログ: {LOG}／出力: {OUT}")
print("学習が完了しました。出力:", OUT, "（評価はまだです）")
